In [1]:
import json
import pandas as pd
import numpy as np
from IPython.display import display

# Đọc file dữ liệu gốc
raw_path = "../data/raw/train_formatted.json"
with open(raw_path, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

print(f"Phiên bản dataset: {raw_data.get('version', 'N/A')}")
print(f"Số lượng bài viết (Articles): {len(raw_data['data'])}")

Phiên bản dataset: viquad2_training_set
Số lượng bài viết (Articles): 138


In [2]:
flattened_data = []

for article in raw_data['data']:
    title = article.get('title', '')
    for p_idx, paragraph in enumerate(article['paragraphs']):
        context = paragraph['context']
        context_id = f"{title}_{p_idx}" # Tạo ID đại diện cho từng đoạn văn
        
        for qa in paragraph['qas']:
            question = qa['question']
            qa_id = qa['id']
            is_impossible = qa.get('is_impossible', False)
            
            # Trích xuất câu trả lời (nếu có)
            answers = qa.get('answers', [])
            if len(answers) > 0:
                answer_text = answers[0]['text']
                answer_start = answers[0]['answer_start']
            else:
                answer_text = ""
                answer_start = -1

            flattened_data.append({
                "qa_id": qa_id,
                "context_id": context_id,
                "title": title,
                "context": context,
                "question": question,
                "answer_text": answer_text,
                "answer_start": answer_start,
                "is_impossible": is_impossible
            })

# Chuyển thành DataFrame để khám phá
df = pd.DataFrame(flattened_data)
print(f"Tổng số mẫu câu hỏi-đáp (QA samples): {len(df)}")
df.head(3)

Tổng số mẫu câu hỏi-đáp (QA samples): 28457


,qa_id,context_id,title,context,question,answer_text,answer_start,is_impossible
0,uit_000001,Phạm Văn Đồng_0,Phạm Văn Đồng,Phạm Văn Đồng (1 tháng 3 năm 1906 – 29 tháng 4...,Tên gọi nào được Phạm Văn Đồng sử dụng khi làm...,Lâm Bá Kiệt,507,False
1,uit_000002,Phạm Văn Đồng_0,Phạm Văn Đồng,Phạm Văn Đồng (1 tháng 3 năm 1906 – 29 tháng 4...,Phạm Văn Đồng giữ chức vụ gì trong bộ máy Nhà ...,Thủ tướng,60,False
2,uit_000003,Phạm Văn Đồng_0,Phạm Văn Đồng,Phạm Văn Đồng (1 tháng 3 năm 1906 – 29 tháng 4...,"Giai đoạn năm 1955-1976, Phạm Văn Đồng nắm giữ...",Thủ tướng Chính phủ Việt Nam Dân chủ Cộng hòa,245,False


In [3]:
print("=== 1. KIỂM TRA GIÁ TRỊ THIẾU (MISSING VALUES) ===")
missing_summary = df.isnull().sum()
print(missing_summary[missing_summary > 0] if missing_summary.sum() > 0 else " Không có giá trị Null/NaN nào trong DataFrame.\n")

# Kiểm tra các chuỗi rỗng ngoài trường answer_text (khi is_impossible=True)
empty_questions = df[df['question'].str.strip() == '']
empty_contexts = df[df['context'].str.strip() == '']
print(f"- Số câu hỏi bị rỗng text: {len(empty_questions)}")
print(f"- Số ngữ cảnh (context) bị rỗng text: {len(empty_contexts)}\n")

=== 1. KIỂM TRA GIÁ TRỊ THIẾU (MISSING VALUES) ===
 Không có giá trị Null/NaN nào trong DataFrame.

- Số câu hỏi bị rỗng text: 0
- Số ngữ cảnh (context) bị rỗng text: 0



In [4]:
print("=== 2. KIỂM TRA DỮ LIỆU TRÙNG LẶP (DUPLICATES) ===")
duplicate_qas = df.duplicated(subset=['qa_id']).sum()
duplicate_pairs = df.duplicated(subset=['context', 'question']).sum()
print(f"- Số trùng lặp QA ID: {duplicate_qas}")
print(f"- Số trùng lặp cặp (Context + Question): {duplicate_pairs}\n")

=== 2. KIỂM TRA DỮ LIỆU TRÙNG LẶP (DUPLICATES) ===
- Số trùng lặp QA ID: 0
- Số trùng lặp cặp (Context + Question): 3



In [5]:
print("=== 3. KIỂM TRA TÍNH HỢP LỆ CỦA ANSWER START ===")
def verify_answer(row):
    if row['is_impossible'] or row['answer_start'] == -1:
        return True
    
    start = row['answer_start']
    text = row['answer_text']
    extracted_text = row['context'][start : start + len(text)]
    return extracted_text == text

df['is_valid_answer'] = df.apply(verify_answer, axis=1)
invalid_answers = df[~df['is_valid_answer']]
print(f"- Số mẫu có `answer_start` bị lệch/không khớp với context: {len(invalid_answers)}")
if len(invalid_answers) > 0:
    print(" Cảnh báo: Cần loại bỏ các mẫu bị lệch vị trí này trước khi train!\n")

=== 3. KIỂM TRA TÍNH HỢP LỆ CỦA ANSWER START ===
- Số mẫu có `answer_start` bị lệch/không khớp với context: 154
 Cảnh báo: Cần loại bỏ các mẫu bị lệch vị trí này trước khi train!



In [6]:
# ==========================================
# 5. THỐNG KÊ CHI TIẾT ĐỘ DÀI & PHÂN PHỐI (DESCRIPTIVE STATS)
# ==========================================
print("=== 4. THỐNG KÊ KÍCH THƯỚC VĂN BẢN (THEO SỐ TỪ) ===")
df['context_word_count'] = df['context'].apply(lambda x: len(x.split()))
df['question_word_count'] = df['question'].apply(lambda x: len(x.split()))
df['answer_word_count'] = df[~df['is_impossible']]['answer_text'].apply(lambda x: len(str(x).split()))

print(f"- Số lượng đoạn văn (Contexts) duy nhất: {df['context_id'].nunique()}")
print(f"- Số câu hỏi không có câu trả lời (is_impossible): {df['is_impossible'].sum()}")
print(f"- Số câu hỏi có câu trả lời: {(~df['is_impossible']).sum()}\n")

stats_df = pd.DataFrame({
    'Context Length': df['context_word_count'].describe(),
    'Question Length': df['question_word_count'].describe(),
    'Answer Length': df['answer_word_count'].describe()
})
display(stats_df)

=== 4. THỐNG KÊ KÍCH THƯỚC VĂN BẢN (THEO SỐ TỪ) ===
- Số lượng đoạn văn (Contexts) duy nhất: 4101
- Số câu hỏi không có câu trả lời (is_impossible): 9217
- Số câu hỏi có câu trả lời: 19240



,Context Length,Question Length,Answer Length
count,28457.000000,28457.000000,19240.000000
mean,180.607724,14.644868,9.953794
std,70.926703,4.977264,10.585834
min,88.000000,1.000000,1.000000
25%,133.000000,11.000000,3.000000
50%,161.000000,14.000000,6.000000
75%,207.000000,18.000000,14.000000
max,1537.000000,53.000000,150.000000


In [7]:
# ==========================================
# THỐNG KÊ ĐỘ DÀI TRUNG BÌNH VÀ TỐI ĐA (THEO TỪ)
# ==========================================
# 1. Tính toán số từ cho từng trường dữ liệu
df["context_word_count"] = df["context"].apply(lambda x: len(str(x).split()))
df["question_word_count"] = df["question"].apply(lambda x: len(str(x).split()))

# Đối với Answer, chỉ tính độ dài trên các câu hỏi có đáp án (is_impossible == False)
answer_lengths = df[~df["is_impossible"]]["answer_text"].apply(
    lambda x: len(str(x).split())
)

# 2. In kết quả theo đúng định dạng
print("--- Độ dài trung bình (tính bằng từ) ---")
print(
    f"Context length  : Trung bình {df['context_word_count'].mean():.1f} từ | Tối đa {df['context_word_count'].max()} từ"
)
print(
    f"Question length : Trung bình {df['question_word_count'].mean():.1f} từ | Tối đa {df['question_word_count'].max()} từ"
)
print(
    f"Answer length   : Trung bình {answer_lengths.mean():.1f} từ | Tối đa {answer_lengths.max()} từ"
)

--- Độ dài trung bình (tính bằng từ) ---
Context length  : Trung bình 180.6 từ | Tối đa 1537 từ
Question length : Trung bình 14.6 từ | Tối đa 53 từ
Answer length   : Trung bình 10.0 từ | Tối đa 150 từ
